# RTA Diagnostic Four-Pack — Excel Input Version

This notebook reads oil production data from an Excel file (Column A = API, Column B = Date, Column C = oil production rate in STB/D) and automatically generates the RTA diagnostic four-pack (log-log rate vs MBT, log-log rate vs time, MBT vs time, and 1/q vs sqrt(t)) for every well/API found in the file.

**Workflow:**
1. Set `excel_file` below and run the notebook top to bottom to get a first-pass set of four-packs for every API.
2. Inspect the plots (each point is labeled with its day number `t`) and note points/time ranges that break monotonically increasing MBT or otherwise look like noise/outliers.
3. Add entries to `cutoff_dict` (max days to keep) and/or `remove_dict` (specific day numbers `t` to drop) for the offending APIs.
4. Re-run the **masking + plotting** cell (near the bottom) — you do not need to reload the Excel file or re-run the whole notebook, just re-run from the `cutoff_dict`/`remove_dict` cell down.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Load the Excel file

Expected layout: **Column A = API**, **Column B = Date** (MM/DD/YYYY), **Column C = oil production rate (STB/D)**.

A header row is assumed (row 1 has column labels). The code below reads the first three columns by *position*, so it doesn't matter what the header text actually says.

In [ ]:
# --- Point this at your file ---
excel_file = "production_data.xlsx"   # <-- change to your file path
sheet_name = 0                          # <-- change if data isn't on the first sheet

raw = pd.read_excel(excel_file, sheet_name=sheet_name, header=0, usecols=[0, 1, 2])

# Standardize column names by position, regardless of what the header text says
raw.columns = ["API", "Date", "qo"]

df = raw.copy()
df["API"] = df["API"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["qo"] = pd.to_numeric(df["qo"], errors="coerce")

# Drop rows that failed to parse
df = df.dropna(subset=["API", "Date"]).copy()

print(f"Loaded {len(df):,} rows across {df['API'].nunique()} unique API(s)")
df.head()


## 2. List of APIs to process

By default this picks up every unique API found in the Excel file (sorted). Edit `api_list` manually if you want to restrict the run to a subset.

In [ ]:
api_list = sorted(df["API"].unique().tolist())
print(f"{len(api_list)} API(s) found:")
api_list


## 3. Core diagnostic calculations

`compute_full_history` builds the full production history for a well (cumulative production `Np`, elapsed time `t`, material balance time `MBT = Np/qo`, `sqrt(t)`, and `1/q`) **once**, from all available data. Nothing here depends on `cutoff_dict`/`remove_dict` — those are applied afterward as a display mask, so the underlying calculation never changes as you clean up the plots.

Because this data doesn't come with an explicit "days on" column, the interval between consecutive readings is estimated from the gap between each date and the one before it (falling back to the median interval in the dataset for the first point). This works for daily, monthly, or irregularly-spaced data.

In [ ]:
def compute_full_history(well_df):
    w = well_df.copy().sort_values("Date")
    w = w[w["qo"] > 0].copy()
    w = w.reset_index(drop=True)

    if len(w) == 0:
        return w

    # interval (days) represented by each reading, inferred from spacing between dates
    diffs = w["Date"].diff().dt.days
    median_interval = diffs.median()
    if pd.isna(median_interval) or median_interval <= 0:
        median_interval = 30.0
    interval_days = diffs.fillna(median_interval).clip(lower=0.0)

    # full-history cumulative production
    w["Np"] = (w["qo"] * interval_days).cumsum()

    # time since first production
    w["t"] = (w["Date"] - w["Date"].iloc[0]).dt.days.astype(float) + 1.0

    # diagnostics computed ONCE from the full history
    w["MBT"] = w["Np"] / w["qo"]
    w["sqrt_t"] = np.sqrt(w["t"])
    w["inv_q"] = 1.0 / w["qo"]

    return w.reset_index(drop=True)


In [ ]:
def apply_plot_mask(well_full, remove_t=None, cutoff_days=None):
    """Filters (masks) the pre-computed full history for plotting/inspection only.
    Does NOT recompute Np/MBT — just hides points."""
    mask = np.ones(len(well_full), dtype=bool)

    if cutoff_days is not None:
        mask &= (well_full["t"] <= cutoff_days)

    if remove_t is not None and len(remove_t) > 0:
        mask &= ~well_full["t"].isin(remove_t)

    return well_full.loc[mask].copy()


In [ ]:
def annotate_points(ax, x, y, df, label_every=1):
    for i in range(0, len(x), label_every):
        ax.annotate(
            f"{int(df['t'].iloc[i])}",
            (x.iloc[i], y.iloc[i]),
            fontsize=7
        )

def plot_fourpack(plot_df, api, title_suffix=""):
    if plot_df.empty:
        print(f"API {api}: nothing left to plot after masking")
        return

    fig, axs = plt.subplots(2, 2, figsize=(14, 10), dpi=150, constrained_layout=True)
    fig.suptitle(f"RTA Diagnostic Four-Pack — API {api}{title_suffix}")

    # log-log Rate vs MBT
    ax = axs[0, 0]
    ax.plot(plot_df["MBT"], plot_df["qo"], marker="o")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("MBT (days)")
    ax.set_ylabel("qo (STB/D)")
    ax.set_title("log-log Rate vs MBT")
    ax.grid(True, which="major", linewidth=0.8)
    ax.grid(True, which="minor", linewidth=0.4)

    # log-log Rate vs Time
    ax = axs[0, 1]
    ax.plot(plot_df["t"], plot_df["qo"], marker="o")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("qo (STB/D)")
    ax.set_title("log-log Rate vs Time")
    ax.grid(True, which="major", linewidth=0.8)
    ax.grid(True, which="minor", linewidth=0.4)

    # MBT vs Time
    ax = axs[1, 0]
    ax.plot(plot_df["t"], plot_df["MBT"], marker="o")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("MBT (days)")
    ax.set_title("MBT vs Time")
    ax.grid(True, which="major", linewidth=0.8)
    ax.grid(True, which="minor", linewidth=0.4)

    # 1/q vs sqrt(t)
    ax = axs[1, 1]
    ax.plot(plot_df["sqrt_t"], plot_df["inv_q"], marker="o")
    ax.set_xlabel("sqrt(t)")
    ax.set_ylabel("1/q")
    ax.set_title("1/q vs sqrt(t)")
    ax.grid(True, which="major", linewidth=0.8)
    ax.grid(True, which="minor", linewidth=0.4)

    annotate_points(ax, plot_df["sqrt_t"], plot_df["inv_q"], plot_df, label_every=1)

    plt.show()


In [ ]:
def run_fourpacks(api_list, cutoff_dict, remove_dict, title_suffix=""):
    """Loops over api_list, applies cutoff_dict/remove_dict masking, and plots the four-pack for each API."""
    for api in api_list:
        well_df = df[df["API"] == api]

        if well_df.empty:
            print(f"API {api}: no data found")
            continue

        well_full = compute_full_history(well_df)

        cutoff_days = cutoff_dict.get(api, None)
        remove_t = remove_dict.get(api, None)

        well_plot = apply_plot_mask(
            well_full,
            remove_t=remove_t,
            cutoff_days=cutoff_days
        )

        plot_fourpack(well_plot, api, title_suffix=title_suffix)


## 4. First pass — all wells, no filtering

Run this once to see the raw diagnostics for every API. Use the day-number labels on each point to identify outliers or a late-time cutoff for each well.

In [ ]:
run_fourpacks(api_list, cutoff_dict={}, remove_dict={}, title_suffix=" (raw, unfiltered)")


## 5. Cutoff / remove dictionaries

Edit these as you inspect the plots above:

- **`cutoff_dict`**: `{API: max_t_days}` — drops every point with `t` beyond `max_t_days` for that API.
- **`remove_dict`**: `{API: {t1, t2, ...}}` — drops the specific day-numbers `t` for that API (e.g. to remove points that break monotonically increasing MBT, shut-in restarts, workovers, etc).

Neither dict recomputes `Np`/`MBT` — they only hide points for plotting, so the underlying full-history calculation stays consistent no matter how you filter.

You can add to these incrementally and re-run the cell below (Section 6) as many times as you like — no need to reload the Excel file or rerun Sections 1–4.

In [ ]:
# Example:
# cutoff_dict = {
#     "4246141255": 1000,
#     "4246138499": 2000,
# }
#
# remove_dict = {
#     "4246141327": {1035, 1217},
#     "4246138996": {1827, 1858, 1888},
# }

cutoff_dict = {
}

remove_dict = {
}


## 6. Re-plot with filtering applied

Re-run this cell any time after editing `cutoff_dict` / `remove_dict` above.

In [ ]:
run_fourpacks(api_list, cutoff_dict=cutoff_dict, remove_dict=remove_dict, title_suffix=" (filtered)")


## 7. Optional: single-well touch-up

To iterate on one well at a time (e.g. while zooming in to read off exact day numbers to add to `remove_dict`), use this instead of re-running the full `api_list` loop.

In [ ]:
single_api = api_list[0]   # <-- change to the API you're working on

run_fourpacks([single_api], cutoff_dict=cutoff_dict, remove_dict=remove_dict, title_suffix=" (single-well check)")
